## Dependencies

In [ ]:
from pathlib import Path
if Path('/kaggle/working').exists(): # Install missing dependencies if run on Kaggle.
    !pip install mlflow contractions --quiet
elif Path('/content').exists(): 
    from google.colab import drive
    drive.mount('/content/drive') # Connect to Google Drive if the notebook is run on Colab.
    !pip install emoji mlflow optuna contractions gensim transformers # Install missing dependencies if run on Colab.
else: 
    pass # manually install requirements.txt file   


In [ ]:
import os
import re
import time
import copy
import html
import json
import random
from datetime import datetime
from collections import Counter, OrderedDict

import emoji
import mlflow
import nltk
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import torch
from contractions import fix
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from tensorflow import keras
from tensorflow.keras.layers import Dense, Embedding, GRU, LSTM
from tensorflow.keras.metrics import FBetaScore
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tqdm import tqdm
from transformers import DistilBertModel, DistilBertTokenizer

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize

from sklearn.base import clone
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, fbeta_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

In [ ]:
FLAGS = re.MULTILINE | re.DOTALL

def ensure_nltk_resource(resource_path, download_name=None):
    """Download an NLTK resource only when it is not already available locally."""
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(download_name or resource_path.split('/')[-1], quiet=True)

ensure_nltk_resource('tokenizers/punkt', 'punkt')
ensure_nltk_resource('tokenizers/punkt_tab', 'punkt_tab')
ensure_nltk_resource('corpora/stopwords', 'stopwords')
ensure_nltk_resource('corpora/wordnet', 'wordnet')
ensure_nltk_resource('corpora/omw-1.4', 'omw-1.4')

def resolve_torch_device(min_cuda_capability=(7, 0)):
    """Return the PyTorch device while falling back to CPU when the available GPU is too old
    for the installed Torch build."""
    if not torch.cuda.is_available():
        return 'cpu', None, 'CUDA is not available for PyTorch in this environment.'

    gpu_name = torch.cuda.get_device_name(0)
    gpu_capability = torch.cuda.get_device_capability(0)
    if gpu_capability < min_cuda_capability:
        message = (
            f'PyTorch will run on CPU because the available GPU ({gpu_name}, sm_{gpu_capability[0]}{gpu_capability[1]}) '
            f'is older than the minimum capability supported by this Torch build (sm_{min_cuda_capability[0]}{min_cuda_capability[1]}).'
        )
        return 'cpu', gpu_name, message

    return 'cuda', gpu_name, 'PyTorch can use the GPU in this environment.'

tf_gpu_available = bool(tf.config.list_physical_devices('GPU'))
device, torch_gpu_name, torch_device_message = resolve_torch_device()

print('TensorFlow version:', tf.__version__)
print('PyTorch version:', torch.__version__)
print('TensorFlow GPU available:', tf_gpu_available)
print('PyTorch CUDA available:', torch.cuda.is_available())
if torch_gpu_name is not None:
    print('Detected GPU for PyTorch:', torch_gpu_name)
print('PyTorch device used by the notebook:', device)
print(torch_device_message)

In [ ]:
RAW_DATA_FILENAME = 'training.1600000.processed.noemoticon.csv'
cwd = Path.cwd().resolve()

# Detect the working environment and declare candidate directories for the raw dataset.
if Path('/kaggle/working').exists():
    RUN_ENV = 'kaggle'
    data_candidates = [
        Path('/kaggle/input/sentiment140'),
        Path('/kaggle/input/datasets/mouhamadoudiagne/sentiment140'),
        Path('/kaggle/input')
    ]
elif Path('/content').exists():
    RUN_ENV = 'colab'
    data_candidates = [
        Path('/content/drive/MyDrive/AI Engineer OC/Projet 7/data/sentiment140'),
        Path('/content/drive/MyDrive/AI Engineer OC/Projet 7'),
        Path('/content')
    ]
else:
    RUN_ENV = 'local'
    data_candidates = [
        cwd / 'data' / 'sentiment140',
        cwd.parent / 'data' / 'sentiment140',
        cwd / 'data',
        cwd.parent / 'data'
    ]

# Expand each candidate so the notebook can work whether the candidate is the dataset root
# itself or a parent directory that contains a `sentiment140` folder.
raw_data_candidates = []
for candidate_dir in data_candidates:
    raw_data_candidates.append(candidate_dir / RAW_DATA_FILENAME)
    raw_data_candidates.append(candidate_dir / 'sentiment140' / RAW_DATA_FILENAME)

RAW_DATA_PATH = next((path for path in raw_data_candidates if path.exists()), None)
if RAW_DATA_PATH is None:
    searched_paths = '\n'.join(f'- {path}' for path in raw_data_candidates)
    raise FileNotFoundError(
        f'Raw dataset not found. The notebook looked for `{RAW_DATA_FILENAME}` in:\n{searched_paths}'
    )

DATA_PATH = RAW_DATA_PATH.parent
print('Working directory :', cwd)
print('Execution environment :', RUN_ENV)
print('Data directory :', DATA_PATH)
print('Raw dataset path :', RAW_DATA_PATH)
print('Raw dataset exists :', RAW_DATA_PATH.exists())

## Data Import

In [ ]:
COLUMN_NAMES = ['target', 'id', 'datetime', 'query', 'username', 'tweet']
SAMPLE_SIZE = 125_000
RANDOM_STATE = 42

# Import the raw Sentiment140 file with the `cp1252` encoding selected during EDA.
tweets = pd.read_csv(
    RAW_DATA_PATH,
    names=COLUMN_NAMES,
    header=None,
    encoding='cp1252'
)

# Keep the raw tweet text for traceability and remap the labels to a binary target for modeling.
tweets['tweet'] = tweets['tweet'].astype(str)
tweets['tweet_raw'] = tweets['tweet']
tweets['target_original'] = tweets['target']
tweets['target'] = tweets['target_original'].replace({0: 0, 4: 1})

# Build a balanced sample to keep the benchmark tractable while preserving class parity.
sample_per_class = SAMPLE_SIZE // tweets['target'].nunique()
data = (
    tweets.groupby('target', group_keys=False)
    .sample(n=sample_per_class, random_state=RANDOM_STATE)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
    .copy()
)

print('Imported rows in full dataset :', len(tweets))
print('Rows kept for benchmarking :', len(data))
print('Unique labels in sample :', sorted(data['target'].unique().tolist()))
data.head()

In [ ]:
# Compare the class balance between the full imported dataframe and the sampled benchmark dataframe.
distribution_summary = pd.DataFrame({
    'full_dataset': tweets['target'].value_counts(normalize=True).sort_index(),
    'sample': data['target'].value_counts(normalize=True).sort_index()
})
distribution_summary.index = distribution_summary.index.map({0: 'negative', 1: 'positive'})
(distribution_summary * 100).round(2)

## Train / Validation / Test Split

In [ ]:
TRAIN_RATIO = 0.6
VALIDATION_RATIO = 0.2
TEST_RATIO = 0.2

assert np.isclose(TRAIN_RATIO + VALIDATION_RATIO + TEST_RATIO, 1.0), 'Split ratios must sum to 1.'

# First separate the training split from the remaining validation/test pool.
data_train, data_temp = train_test_split(
    data,
    train_size=TRAIN_RATIO,
    stratify=data['target'],
    random_state=RANDOM_STATE
)

# Then split the remaining pool equally to recover the requested 0.2 / 0.2 proportions.
validation_share_of_remaining = VALIDATION_RATIO / (VALIDATION_RATIO + TEST_RATIO)
data_val, data_test = train_test_split(
    data_temp,
    train_size=validation_share_of_remaining,
    stratify=data_temp['target'],
    random_state=RANDOM_STATE
)

# Reset indexes so later preprocessing steps can work on clean standalone dataframes.
data_train = data_train.reset_index(drop=True).copy()
data_val = data_val.reset_index(drop=True).copy()
data_test = data_test.reset_index(drop=True).copy()

print('Train shape :', data_train.shape)
print('Validation shape :', data_val.shape)
print('Test shape :', data_test.shape)

In [ ]:
# Verify both split sizes and class proportions before benchmarking.
split_distribution = pd.DataFrame({
    'train_rows': [len(data_train)],
    'validation_rows': [len(data_val)],
    'test_rows': [len(data_test)],
    'train_negative_pct': [round((data_train['target'] == 0).mean() * 100, 2)],
    'train_positive_pct': [round((data_train['target'] == 1).mean() * 100, 2)],
    'validation_negative_pct': [round((data_val['target'] == 0).mean() * 100, 2)],
    'validation_positive_pct': [round((data_val['target'] == 1).mean() * 100, 2)],
    'test_negative_pct': [round((data_test['target'] == 0).mean() * 100, 2)],
    'test_positive_pct': [round((data_test['target'] == 1).mean() * 100, 2)]
})
split_distribution

## Performance Metrics

In [ ]:
BADBUZZ_REFERENCE_VOLUME = 1000
NEGATIVE_LABEL = 0

def badbuzz_miss_rate(y_true, y_pred, negative_label=NEGATIVE_LABEL, reference_volume=BADBUZZ_REFERENCE_VOLUME):
    """Estimate how many truly negative tweets would be predicted as non-negative
    over a reference volume of negative tweets."""
    labels = sorted(pd.Index(y_true).append(pd.Index(y_pred)).unique().tolist())
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    label_to_index = {label: idx for idx, label in enumerate(labels)}

    if negative_label not in label_to_index:
        return np.nan

    neg_idx = label_to_index[negative_label]
    true_negatives = cm[neg_idx, neg_idx]
    false_positives = cm[neg_idx].sum() - true_negatives
    total_negative = true_negatives + false_positives

    if total_negative == 0:
        return np.nan

    return round(reference_volume * false_positives / total_negative)

def evaluate_predictions(y_true, y_pred, dataset_name='dataset', beta=3.0):
    """Compute the benchmark metrics used throughout the notebook for one set of predictions."""
    metrics_summary = pd.Series({
        'dataset': dataset_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_negative': precision_score(y_true, y_pred, pos_label=NEGATIVE_LABEL, average='binary', zero_division=0),
        'recall_negative': recall_score(y_true, y_pred, pos_label=NEGATIVE_LABEL, average='binary', zero_division=0),
        'fbeta_negative': fbeta_score(y_true, y_pred, beta=beta, pos_label=NEGATIVE_LABEL, average='binary', zero_division=0),
        f'badbuzz_miss_per_{BADBUZZ_REFERENCE_VOLUME}_negative_tweets': badbuzz_miss_rate(y_true, y_pred)
    })
    return metrics_summary.to_frame().T

def plot_confusion_matrix_from_predictions(y_true, y_pred, title='Confusion Matrix'):
    """Display a confusion matrix heatmap for a fitted model's predictions."""
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    cm_df = pd.DataFrame(
        cm,
        index=['true_negative', 'true_positive'],
        columns=['pred_negative', 'pred_positive']
    )
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.show()

In [ ]:
# Small sanity check on handcrafted predictions so the metric definitions stay transparent.
y_true_example = pd.Series([0, 0, 0, 1, 1, 1])
y_pred_example = pd.Series([0, 1, 0, 1, 0, 1])

display(evaluate_predictions(y_true_example, y_pred_example, dataset_name='sanity_check'))
plot_confusion_matrix_from_predictions(y_true_example, y_pred_example, title='Performance metric sanity check')